# AMOEBA on the PYNQ-Z2

Drives the RV64 soft core in the PL from the Cortex-A9.

**PYNQ's Jupyter kernel runs as root**, so `/dev/mem` works here without `sudo` —
which is the main reason a notebook is more pleasant than SSH for poking at the
board interactively. For scripted or regression use, `run_freertos.py` over SSH
is the better tool.

In [ ]:
import sys, time
sys.path.insert(0, '/home/xilinx/amoeba/sw')

from amoeba import Amoeba, image, regs

## 1. Program the PL and check we are talking to it

`ID` reading `AMOB` is the single most valuable early check: a wrong value means
the bitstream did not load or the base address is wrong, and *every* later
symptom — a core that never runs, an empty console — is downstream of it.

Note this uses `Bitstream`, not `Overlay`. The driver addresses the control block
at a fixed address from `regs.py` rather than discovering it, so the `.hwh` is not
needed to run FreeRTOS. It is needed later, for the trace DMA.

In [ ]:
dev = Amoeba(bitstream='/home/xilinx/amoeba/amoeba.bit')
print(dev.describe())

## 2. Load a program

The core is held in reset out of configuration, so this happens in a quiescent
design. `load()` refuses to write while the core runs, refuses an image that does
not fit, and checks the ELF's `tohost` symbol against the address the bus monitor
watches — that last one prevents a run that prints perfect console output and then
hangs forever.

In [ ]:
img = image.load('/home/xilinx/amoeba/images/freertos_wally.elf')
print(f'{img.load_base:#010x}..{img.load_end:#010x}  {img.file_bytes} bytes  '
      f'entry {img.entry:#010x}  tohost {img.tohost():#010x}')

dev.run(img)
print('running, in_reset =', dev.in_reset)

## 3. Watch the heartbeat

Runs for 10 seconds and prints the console as it arrives.

In [ ]:
t0, c0 = time.monotonic(), dev.cycles
for chunk in dev.stream_console(10.0, until_tohost=False):
    print(chunk.decode('ascii', 'replace'), end='')

wall  = time.monotonic() - t0
cycles = dev.cycles - c0
print(f'\n--- {wall:.2f} s, {cycles} cycles, {dev.retired} retired, {dev.traps} traps')
print(f'--- fabric clock: {cycles/wall/1e6:.3f} MHz')

## 4. Are the beats on time?

The check that nothing else can do. The guest schedules
`configCPU_CLOCK_HZ x 0.1` mtime increments per beat and mtime advances once per
fabric clock, so beats arrive late by exactly `configCPU_CLOCK_HZ / FCLK`. A 4x
wrong constant gives 400 ms beats — while every terminating test still passes,
because they check ordering and results, not rates.

In [ ]:
import re
beats, t = [], time.monotonic()
buf = b''
deadline = t + 5.0
while time.monotonic() < deadline:
    buf += dev.read_console()
    while b'\n' in buf:
        line, _, buf = buf.partition(b'\n')
        if re.search(rb'HB seq=', line):
            beats.append(time.monotonic())
    time.sleep(0.002)

gaps = [b - a for a, b in zip(beats, beats[1:])]
if len(gaps) >= 2:
    mean = sum(gaps) / len(gaps)
    print(f'{len(beats)} beats, mean {mean*1e3:.2f} ms, '
          f'jitter {(max(gaps)-min(gaps))*1e3:.2f} ms')
    print('calibration', 'OK' if abs(mean - 0.100) < 0.005 else
          f'OFF by {mean/0.100:.2f}x -- rebuild with a corrected FPGA_CLOCK_HZ')
else:
    print('not enough beats; is the heartbeat image loaded?')

## 5. Run a terminating test

Same flow, but wait for the HTIF exit instead of a fixed time.

In [ ]:
img = image.load('/home/xilinx/amoeba/images/tc_semaphore.elf')
dev.run(img)

for chunk in dev.stream_console(30.0):
    print(chunk.decode('ascii', 'replace'), end='')

print('tohost seen:', dev.tohost_valid)
print('exit code  :', dev.exit_code if dev.tohost_valid else '(timeout)')
print('retired    :', dev.retired, ' traps:', dev.traps)

`retired` is the number to compare against simulation. This configuration is
deterministic from reset — no DDR refresh, no external interrupts, one clock — so
it should match the simulator's retired-instruction count *exactly*. `cycles` will
not, and should not: simulation memory is 3 cycles, block RAM is 1.